In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/Users/vayunandan/Documents/sentiment-analysis/data/data.csv')

In [ ]:
df.head

In [ ]:
df_copy = df.copy()

In [ ]:
df_copy.drop('language', axis=1, inplace = True)

In [ ]:
df_copy.head

In [ ]:
new_sentiment = {
    "5 stars" : 1,
    "4 stars" : 1,
    "3 stars" : 0,
    "2 stars" : -1,
    "1 star" : -1
}

df_copy['new_sentiment'] = df_copy['sentiment'].map(new_sentiment)

In [ ]:
df_copy.head
df_copy = df_copy.drop(columns='sentiment', axis = 1)

In [ ]:
df_copy = df_copy.rename(columns={'new_sentiment' : 'labels'})

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_copy,
    test_size = 0.2,
    stratify = df_copy['labels']
)

In [ ]:
train_df.head(5)

In [ ]:
len(train_df)

In [ ]:
len(temp_df)

In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['labels']
)

In [ ]:
import matplotlib as plt
import plotly.express as px
import plotly.offline as pyo
import plotly.graph_objects as go

df_counts = train_df['new_sentiment'].value_counts().reset_index()
df_counts.columns = ['sentiment_value', 'counts']

# 2. Map the integer values to strings explicitly
label_map = {-1: 'Negative', 0: 'Neutral', 1: 'Positive'}
df_counts['label'] = df_counts['sentiment_value'].map(label_map)

# 3. Plot using the DataFrame columns
# This ensures color '1' always stays with 'Positive', etc.
fig = px.bar(df_counts, 
             x='label', 
             y='counts',
             color='sentiment_value',
             color_continuous_scale='Viridis', # Or your Dark24 sequence
             title='<b>Sentiments Counts</b>')

fig.update_layout(xaxis_title='Sentiment',
                  yaxis_title='Counts',
                  template='plotly_dark',
                  coloraxis_showscale=False) # Hides the color bar on the right

fig.show()
pyo.plot(fig, filename='Sentiments_Counts.html', auto_open=True)

In [ ]:
train_pos = train_df[train_df['labels']==-1]
len(train_pos)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd

# Load your data (assuming you already have train_df and val_df)
# train_df should have columns like 'text' and 'label'
label_map = {-1: 0, 0: 1, 1: 2}
train_df['label'] = train_df['labels'].map(label_map)

# Convert pandas DataFrames to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)  # if you have validation data

# Load model and tokenizer
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)  # adjust num_labels

model.gradient_checkpointing_enable()
# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples['tweet'], padding='max_length', truncation=True, max_length=64)

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=6,
    gradient_accumulation_steps= 2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    use_mps_device=True  # M1 GPU acceleration
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train!
trainer.train()